# Case Study 12: Telco Customer Churn Prediction
## Aurora-GLM Showcase: Binomial GLM (Logistic Regression)

---

## Overview

This notebook demonstrates **Binomial GLM** (logistic regression) for binary classification. We predict customer churn using demographic and service features.

### Research Questions

1. Which factors predict customer churn?
2. How to interpret odds ratios?
3. What is the predictive accuracy?

### Aurora-GLM Capabilities

1. Binomial GLM with logit link
2. Odds ratio interpretation
3. ROC curves and AUC
4. Multi-backend comparison

---

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import time
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from aurora.models.glm import fit_glm

try:
    import torch
    TORCH_AVAILABLE = True
    GPU_AVAILABLE = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'N/A'
except ImportError:
    TORCH_AVAILABLE = False
    GPU_AVAILABLE = False
    GPU_NAME = 'N/A'

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context('notebook', font_scale=1.1)
%config InlineBackend.figure_format = 'retina'
np.random.seed(42)

print("="*80)
print("ENVIRONMENT SETUP")
print("="*80)
print(f"PyTorch: {'Available' if TORCH_AVAILABLE else 'Not installed'}")
print(f"GPU: {'Available - ' + GPU_NAME if GPU_AVAILABLE else 'Not available'}")
print("="*80)

In [ ]:
# Load Telco Churn data
data_path = Path('data/telco_churn.csv')

if not data_path.exists():
    print("Downloading Telco Churn data...")
    try:
        import openml
        dataset = openml.datasets.get_dataset(42178)
        df, _, _, _ = dataset.get_data()
        data_path.parent.mkdir(exist_ok=True)
        df.to_csv(data_path, index=False)
    except:
        import urllib.request
        url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
        data_path.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve(url, data_path)
    print(f"Downloaded to {data_path}")

df = pd.read_csv(data_path)

print("="*80)
print("DATA LOADING")
print("="*80)
print(f"Observations: {len(df):,}")
print(f"Variables: {len(df.columns)}")

# Calculate churn rate properly
churn_counts = df['Churn'].value_counts(normalize=True)
if 'Yes' in churn_counts.index:
    churn_rate = churn_counts['Yes'] * 100
elif 1 in churn_counts.index:
    churn_rate = churn_counts[1] * 100
else:
    churn_rate = churn_counts.iloc[1] * 100 if len(churn_counts) > 1 else 0
print(f"\nChurn Rate: {churn_rate:.1f}%")
print("="*80)

In [ ]:
# Preprocessing
print("="*80)
print("PREPROCESSING")
print("="*80)

# Convert target to binary
if df['Churn'].dtype == 'object':
    df['churn'] = (df['Churn'] == 'Yes').astype(int)
else:
    df['churn'] = df['Churn'].astype(int)

# Convert TotalCharges to numeric
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Select key predictors
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges'] if 'TotalCharges' in df.columns else ['tenure', 'MonthlyCharges']
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'SeniorCitizen']

# Standardize numeric
for col in numeric_cols:
    if col in df.columns:
        df[f'{col}_std'] = (df[col] - df[col].mean()) / df[col].std()

# Create binary dummies
df['male'] = (df['gender'] == 'Male').astype(int) if 'gender' in df.columns else 0
df['partner'] = (df['Partner'] == 'Yes').astype(int) if 'Partner' in df.columns else 0
df['dependents'] = (df['Dependents'] == 'Yes').astype(int) if 'Dependents' in df.columns else 0
df['paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int) if 'PaperlessBilling' in df.columns else 0
df['senior'] = df['SeniorCitizen'].astype(int) if 'SeniorCitizen' in df.columns else 0

# Contract type
if 'Contract' in df.columns:
    df['contract_1yr'] = (df['Contract'] == 'One year').astype(int)
    df['contract_2yr'] = (df['Contract'] == 'Two year').astype(int)

print(f"Target: churn (0/1)")
print(f"Churn rate: {df['churn'].mean()*100:.1f}%")
print(f"Numeric predictors: {numeric_cols}")
print("="*80)

In [ ]:
# EDA
print("="*80)
print("EXPLORATORY DATA ANALYSIS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Churn distribution
df['churn'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['steelblue', 'coral'])
axes[0, 0].set_title('Churn Distribution', fontweight='bold')
axes[0, 0].set_xticklabels(['No', 'Yes'], rotation=0)

# Tenure by churn
if 'tenure' in df.columns:
    df.boxplot(column='tenure', by='churn', ax=axes[0, 1])
    axes[0, 1].set_title('Tenure by Churn', fontweight='bold')
    plt.suptitle('')

# Monthly charges by churn
if 'MonthlyCharges' in df.columns:
    df.boxplot(column='MonthlyCharges', by='churn', ax=axes[1, 0])
    axes[1, 0].set_title('Monthly Charges by Churn', fontweight='bold')
    plt.suptitle('')

# Contract type by churn
if 'Contract' in df.columns:
    contract_churn = df.groupby('Contract')['churn'].mean() * 100
    contract_churn.plot(kind='bar', ax=axes[1, 1], color='purple', alpha=0.7)
    axes[1, 1].set_title('Churn Rate by Contract Type', fontweight='bold')
    axes[1, 1].set_ylabel('Churn Rate (%)')
    axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print("="*80)

## Mathematical Specification

### Binomial GLM (Logistic Regression)

$$Y_i \sim \text{Bernoulli}(\pi_i)$$

$$\text{logit}(\pi_i) = \log\left(\frac{\pi_i}{1-\pi_i}\right) = \mathbf{x}_i^T \boldsymbol{\beta}$$

### Odds Ratio Interpretation

$$\text{Odds Ratio} = e^{\beta_j}$$

- OR > 1: Increased odds of churn
- OR < 1: Decreased odds of churn

In [ ]:
# Prepare design matrix
print("="*80)
print("MODEL FITTING")
print("="*80)

# Build feature matrix
features = [np.ones(len(df))]  # Intercept
names = ['Intercept']

if 'tenure_std' in df.columns:
    features.append(df['tenure_std'].values)
    names.append('Tenure')

if 'MonthlyCharges_std' in df.columns:
    features.append(df['MonthlyCharges_std'].values)
    names.append('MonthlyCharges')

for col in ['senior', 'partner', 'dependents', 'paperless']:
    if col in df.columns:
        features.append(df[col].values)
        names.append(col.capitalize())

if 'contract_1yr' in df.columns:
    features.append(df['contract_1yr'].values)
    features.append(df['contract_2yr'].values)
    names.extend(['Contract_1yr', 'Contract_2yr'])

X = np.column_stack(features)
y = df['churn'].values

print(f"Design Matrix: {X.shape}")
print(f"Predictors: {names}")

# Fit model
start_time = time.time()
result = fit_glm(X=X, y=y, family='binomial', link='logit')
time_fit = time.time() - start_time

print(f"\nConverged: {result.converged_}")
print(f"Iterations: {result.n_iter_}")
print(f"Time: {time_fit:.3f}s")
print(f"\nAIC: {result.aic_:.2f}")
print(f"BIC: {result.bic_:.2f}")
print("="*80)

In [ ]:
# Odds ratios
print("="*80)
print("ODDS RATIOS")
print("="*80)

print("\nCoefficients and Odds Ratios:")
print("-" * 60)

for name, coef in zip(names, result.coef_):
    odds_ratio = np.exp(coef)
    if name == 'Intercept':
        print(f"{name:20s}: coef={coef:+.4f}")
    else:
        direction = 'increases' if coef > 0 else 'decreases'
        print(f"{name:20s}: coef={coef:+.4f}, OR={odds_ratio:.3f}")

print("\n" + "="*80)

In [ ]:
# Predictions and ROC
print("="*80)
print("PREDICTIVE PERFORMANCE")
print("="*80)

# Predicted probabilities
eta = X @ result.coef_
prob = 1 / (1 + np.exp(-eta))

# Classification at 0.5 threshold
pred_class = (prob > 0.5).astype(int)
accuracy = (pred_class == y).mean()

# Confusion matrix
tp = ((pred_class == 1) & (y == 1)).sum()
tn = ((pred_class == 0) & (y == 0)).sum()
fp = ((pred_class == 1) & (y == 0)).sum()
fn = ((pred_class == 0) & (y == 1)).sum()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0

print(f"\nClassification Metrics (threshold=0.5):")
print(f"   Accuracy: {accuracy:.3f}")
print(f"   Sensitivity: {sensitivity:.3f}")
print(f"   Specificity: {specificity:.3f}")
print(f"   Precision: {precision:.3f}")

# ROC curve
thresholds = np.linspace(0, 1, 100)
tpr_list = []
fpr_list = []
for t in thresholds:
    pred = (prob > t).astype(int)
    tpr = ((pred == 1) & (y == 1)).sum() / max(1, (y == 1).sum())
    fpr = ((pred == 1) & (y == 0)).sum() / max(1, (y == 0).sum())
    tpr_list.append(tpr)
    fpr_list.append(fpr)

# AUC (trapezoidal)
auc = -np.trapz(tpr_list, fpr_list)
print(f"   AUC: {auc:.3f}")

# Plot ROC
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_list, tpr_list, 'b-', linewidth=2, label=f'ROC (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print("="*80)

In [ ]:
# Multi-backend benchmark
print("="*80)
print("MULTI-BACKEND PERFORMANCE")
print("="*80)

results = [{'Backend': 'NumPy', 'Time': f'{time_fit:.3f}s'}]

if TORCH_AVAILABLE:
    start = time.time()
    _ = fit_glm(X=X, y=y, family='binomial', link='logit', backend='torch', device='cpu')
    results.append({'Backend': 'PyTorch CPU', 'Time': f'{time.time()-start:.3f}s'})

if GPU_AVAILABLE:
    start = time.time()
    _ = fit_glm(X=X, y=y, family='binomial', link='logit', backend='torch', device='cuda')
    results.append({'Backend': 'PyTorch GPU', 'Time': f'{time.time()-start:.3f}s'})

print(pd.DataFrame(results).to_string(index=False))
print("="*80)

## Conclusions

### Key Findings
1. **Tenure** is protective - longer customers less likely to churn
2. **Monthly charges** increase churn risk
3. **Contract type** strongly protective (1-2 year contracts)
4. **Paperless billing** associated with higher churn

### Aurora-GLM Capabilities
- Binomial GLM with logit link
- Odds ratio interpretation
- ROC/AUC evaluation
- Multi-backend support

---
**Dataset**: Telco Customer Churn (OpenML/IBM)